# Setup
## Dependencies

In [ ]:
%config InlineBackend.figure_format = 'retina'
from pathlib import Path
import sqlite3
from astropy.io import fits
from astropy.time import Time
from astropy.coordinates import SkyCoord
from snpy import get_sn

## File Paths

In [ ]:
db_path = Path("../databases/Master_Database.db").resolve()
conn = sqlite3.connect(db_path)

mosaic_folder = Path("../output mosaics/flats")
output_folder = Path("../output snoopy")

max_mag_err = 0.3

## Dictionaries

In [ ]:
snoopy_targets = {
    "SN2017HHZ": 0.0392,   # host z 0.034
    "SN2017HN": 0.0238,
    "SN2017MS": 0.0246,
    "SN2018GV": 0.0053,
    "SN2018KSS": 0.021,    # light curve contaminated by host nucleus
    "SN2019AIK": 0.023,
    "SN2019NP": 0.00452,   # was 0.004252, a typo
    "SN2019TGM": 0.019107,
    "SN2020SE": 0.031789,
    "SN2020UEC": 0.016,
    "SN2020VEY": 0.01787,
    "SN2020WCJ": 0.0237,
    "SN2021XN": 0.017,     # host z 0.013
    "SN2024ANA": 0.02005,
    "SN2024XYN": 0.021522,
    "SN2024YM": 0.038,
    "SN2025AHKT": 0.026228,
    "SN2025AHSA": 0.015,
    "SN2026AAIV": 0.002722,
    "SN2026ACNA": 0.017365,
    "SN2026AEIV": 0.0078,  # 91bg-like, check SNooPy model
    "SN2026AESJ": 0.019095,   # only 7 groups
    "SN2026AFNJ": 0.024,      # only 2 groups, no photometry yet
    "SN2026AGUA": 0.05,       # only 2 groups, no photometry yet
    "SN2026AGUC": 0.035,      # 91T-like, z from TNS spectra, 4 groups, no photometry yet
    "SN2026AHVI": 0.006997,   # 9 groups, no photometry yet
    "SN2026AICD": 0.025,      # 12 groups, no photometry yet
    "SN2026ATB": 0.017941,
    "SN2026GM": 0.0185,
    "SN2026KC": 0.0228,
    "SN2026AKBO": 0.0173,     # host z from NED via TNS classification, 4 groups (pt5m BVRI), no photometry yet
}

In [ ]:
sn_coordinates = {
    # 2017
    "7hhz": ("01h44m16.910s", "+12d15m18.54s"),   # SN 2017hhz, SN Ia
    "17hn": ("13h07m39.419s", "+06d20m15.23s"),   # SN 2017hn, SN Ia
    "17ms": ("10h26m42.360s", "+36d40m49.76s"),   # SN 2017ms, SN Ia
    # 2018
    "18gv": ("08h05m34.580s", "-11d26m16.87s"),   # SN 2018gv, SN Ia
    "8kss": ("06h46m23.860s", "+43d49m27.80s"),   # SN 2018kss, SN Ia
    # 2019
    "9aik": ("10h37m38.390s", "+37d27m29.56s"),   # SN 2019aik, SN Ia
    "19np": ("10h29m21.980s", "+29d30m38.30s"),   # SN 2019np, SN Ia
    "9tgm": ("22h18m10.970s", "+45d43m10.13s"),   # SN 2019tgm, SN Ia
    # 2020
    "20se": ("03h37m48.700s", "+03d16m19.56s"),   # SN 2020se, SN Ia
    "0uec": ("21h09m48.530s", "+15d08m54.89s"),   # SN 2020uec, SN Ia
    "0vey": ("04h01m16.150s", "+31d44m03.11s"),   # SN 2020vey, SN Ia
    "0wcj": ("02h52m50.310s", "-01d13m51.71s"),   # SN 2020wcj, SN Ia
    # 2021
    "21xn": ("01h37m50.930s", "+53d39m51.91s"),   # SN 2021xn, SN Ia
    # 2024
    "4ana": ("12h58m22.578s", "+27d57m58.14s"),   # SN 2024ana, SN Ia
    "4xyn": ("02h39m23.969s", "+02d39m07.88s"),   # SN 2024xyn, SN Ia
    "24ym": ("10h11m37.390s", "+74d59m38.15s"),   # SN 2024ym, SN Ia
    # 2025
    "ahkt": ("09h25m42.441s", "+22d20m05.76s"),   # SN 2025ahkt, SN Ia
    "ahsa": ("14h35m00.210s", "+00d59m10.38s"),   # SN 2025ahsa, SN Ia
    # 2026
    "aaiv": ("22h37m05.628s", "+34d24m35.19s"),   # SN 2026aaiv, SN Ia
    "abxl": ("13h23m45.388s", "+31d33m52.17s"),   # SN 2026abxl, SN Ia, not in Master_Database yet
    "acna": ("01h48m51.954s", "+10d35m11.40s"),   # SN 2026acna, SN Ia
    "acpp": ("09h48m42.485s", "+44d04m27.95s"),   # SN 2026acpp, SN IIb
    "aeiq": ("08h55m23.301s", "-07d33m40.54s"),   # SN 2026aeiq, SN Ic
    "aeiv": ("03h16m41.828s", "+80d48m09.12s"),   # SN 2026aeiv, SN Ia-91bg-like
    "aeiz": ("00h55m17.687s", "+49d59m50.14s"),   # AT 2026aeiz, unclassified
    "aepk": ("02h09m55.997s", "+10d45m36.17s"),   # SN 2026aepk, SN Iax[02cx-like]
    "aesj": ("07h48m40.024s", "+54d36m43.42s"),   # SN 2026aesj, SN Ia
    "afnj": ("02h48m41.104s", "+14d18m22.41s"),   # SN 2026afnj, SN Ia
    "afrp": ("04h50m52.231s", "-04d53m35.15s"),   # SN 2026afrp, SN Ib
    "afsg": ("12h02m02.958s", "+29d50m40.95s"),   # SN 2026afsg, SN Ib/c, not in Master_Database yet
    "afsy": ("23h33m45.021s", "+25d47m03.53s"),   # SN 2026afsy, SN
    "afys": ("04h13m54.525s", "+29d09m15.91s"),   # AT 2026afys, unclassified
    "agua": ("05h05m54.591s", "+10d10m52.74s"),   # SN 2026agua, SN Ia
    "aguc": ("04h38m04.131s", "-10d52m19.55s"),   # SN 2026aguc, SN Ia-91T-like
    "aguv": ("07h37m04.549s", "+65d35m42.56s"),   # AT 2026aguv, unclassified
    "ahab": ("09h11m02.822s", "+13d24m57.94s"),   # AT 2026ahab, unclassified
    "ahvi": ("12h13m50.443s", "+13d25m26.24s"),   # SN 2026ahvi, SN Ia
    "aicd": ("23h58m54.253s", "+12d08m52.21s"),   # SN 2026aicd, SN Ia
    "6atb": ("14h34m27.294s", "+21d56m58.85s"),   # SN 2026atb, SN Ia
    "26gm": ("04h24m37.413s", "+33d51m43.60s"),   # SN 2026gm, SN Ia
    "26kc": ("10h22m25.076s", "+04d00m01.56s"),   # SN 2026kc, SN Ia
    "akbo": ("10h39m38.911s", "+25d19m13.52s"),   # SN 2026akbo, SN Ia
}

In [ ]:
snoopy_filters = {
    "B": "Bs",
    "V": "Vs",
    "R": "Rs",
    "I": "Is",
    "r": "Rs",
}

In [ ]:
galactic_reddening = {
    "SN2017hhz": 0.0402,
    "SN2017hn": 0.0286,
    "SN2017ms": 0.0088,
    "SN2018gv": 0.0497,
    "SN2018kss": 0.0869,
    "SN2019aik": 0.0180,
    "SN2019np": 0.0172,
    "SN2019tgm": 0.1774,
    "SN2020se": 0.1299,
    "SN2020uec": 0.0731,
    "SN2020vey": 0.2354,
    "SN2020wcj": 0.0555,
    "SN2021xn": 0.2626,
    "SN2024ana": 0.0095,
    "SN2024xyn": 0.0287,
    "SN2024ym": 0.0228,
    "SN2025ahkt": 0.0226,
    "SN2025ahsa": 0.0380,
    "SN2026aaiv": 0.0782,
    "SN2026acna": 0.1065,
    "SN2026aeiv": 0.2032,
    "SN2026aesj": 0.0518,
    "SN2026afnj": 0.0891,
    "SN2026agua": 0.1886,
    "SN2026aguc": 0.0612,
    "SN2026ahvi": 0.0282,
    "SN2026aicd": 0.0836,
    "SN2026atb": 0.0466,
    "SN2026gm": 0.3443,
    "SN2026kc": 0.0283,
    "SN2026akbo": 0.0213,
}

# Functions

In [ ]:
def get_target_data(conn, object_name):
    sql = """
        SELECT filter, mosaic_name_f, mag_f, mag_err_f
        FROM observations
        WHERE SUBSTR(object, 3) = SUBSTR(?, 3)
            AND mag_f IS NOT NULL
        ORDER BY obs_date
    """
    
    output = conn.execute(sql, (object_name,)).fetchall()
    return output

In [ ]:
def convert_mjd(mosaic_path):
    header = fits.getheader(mosaic_path)
    start_time = Time(header["DATE-OBS"], format="isot", scale="utc")
    return start_time.mjd

In [ ]:
def write_snoopy_file(conn, object_name, redshift, output_folder):
    rows = get_target_data(conn, object_name)

    short_sn_name = object_name[-4:].lower()
    ra_text, dec_text = sn_coordinates[short_sn_name]
    position = SkyCoord(ra_text, dec_text)
    ra_deg = position.ra.deg
    dec_deg = position.dec.deg


    snoopy_name = object_name[:6] + object_name[6:].lower()

    points = {"Bs": [], "Vs": [], "Rs": [], "Is": []}
    skipped_band = 0
    skipped_error = 0

    for band, mosaic_name, mag, mag_err in rows:
        if band not in snoopy_filters:
            skipped_band = skipped_band + 1
            continue

        if mag_err is None or mag_err > max_mag_err:
            skipped_error = skipped_error + 1
            continue

        mjd = convert_mjd(mosaic_folder / mosaic_name)
        filter_name = snoopy_filters[band]
        points[filter_name].append((mjd, mag, mag_err))

    lines = [f"{snoopy_name} {redshift} {ra_deg:.6f} {dec_deg:.6f}"]

    for filter_name in points:
        filter_points = sorted(points[filter_name])
        if len(filter_points) == 0:
            continue

        lines.append(f"filter {filter_name}")
        for mjd, mag, mag_err in filter_points:
            lines.append(f"{mjd:.5f}  {mag:.3f}  {mag_err:.3f}")

    output_folder.mkdir(parents=True, exist_ok=True)
    output_file = output_folder / f"{snoopy_name}_snoopy.txt"
    output_file.write_text("\n".join(lines) + "\n")

    print(snoopy_name, "->", output_file.name)
    for filter_name in points:
        print("   ", filter_name, len(points[filter_name]), "points")
    print("    skipped (r or C band):", skipped_band)
    print("    skipped (error too big):", skipped_error)

    return output_file

# Export SNooPy Files

In [ ]:
for object_name in snoopy_targets:
    redshift = snoopy_targets[object_name]
    write_snoopy_file(conn, object_name, redshift, output_folder)

# SNooPy Fit
`get_sn` first tries to read the file as a saved SNooPy object, so it prints
"could not load pickle file ... trying as plain text import". That is normal.

In [ ]:
def get_snpy(txt_file):
    sn_file = output_folder / txt_file

    s = get_sn(str(sn_file))
    s.EBVgal = galactic_reddening[s.name]

    s.choose_model("EBV_model2", stype="st")
    s.fit()
    s.summary()

    s.plot(outfile=str(output_folder / "snpy_plots" / f"{s.name}_fit.pdf"))

    with (output_folder / "snpy_summaries" / f"{s.name}_summary.txt").open("w") as file:
        s.summary(out=file)


    s.save(str(output_folder / "snpy_files" / f"{s.name}.snpy"))
    

In [ ]:
for file in sorted(Path("../output snoopy").glob("*_snoopy.txt")):
    try:
        get_snpy(file)
    except Exception as err:
        print("FAILED", file.name, "-", type(err).__name__, err)

In [ ]:
sn_file = output_folder / "SN2026aaiv_snoopy.txt"
#sn_file = output_folder / "SN2026aaiv.snpy"

s = get_sn(str(sn_file))
s.EBVgal = galactic_reddening[s.name]

s.choose_model("EBV_model2", stype="st")
s.fit()
s.summary()

s.plot(outfile=str(output_folder / f"{s.name}_fit.pdf"))

with (output_folder / f"{s.name}_summary.txt").open("w") as file:
    s.summary(out=file)


s.save(str(output_folder / f"{s.name}.snpy"))

In [ ]:
sn_file = output_folder / "SN2026acna_snoopy.txt"
#sn_file = output_folder / "SN2026acna.snpy"

s = get_sn(str(sn_file))
s.EBVgal = galactic_reddening[s.name]

s.choose_model("EBV_model2", stype="st")
s.fit()
s.summary()

s.plot(outfile=str(output_folder / f"{s.name}_fit.pdf"))

with (output_folder / f"{s.name}_summary.txt").open("w") as file:
    s.summary(out=file)

s.save(str(output_folder / f"{s.name}.snpy"))

In [ ]:
sn_file = output_folder / "SN2026aeiv_snoopy.txt"
#sn_file = output_folder / "SN2026aeiv.snpy"

s = get_sn(str(sn_file))
s.EBVgal = galactic_reddening[s.name]

s.choose_model("EBV_model2", stype="st")
s.fit()
s.summary()


s.plot(outfile=str(output_folder / f"{s.name}_fit.pdf"))

with (output_folder / f"{s.name}_summary.txt").open("w") as file:
    s.summary(out=file)
    

s.save(str(output_folder / f"{s.name}.snpy"))

In [ ]:
sn_file = output_folder / "SN2019np_snoopy.txt"
#sn_file = output_folder / "SN2026aaiv.snpy"

s = get_sn(str(sn_file))
s.EBVgal = galactic_reddening[s.name]

s.choose_model("EBV_model2", stype="st")
s.fit()
s.summary()

s.plot(outfile=str(output_folder / f"{s.name}_fit.pdf"))

with (output_folder / f"{s.name}_summary.txt").open("w") as file:
    s.summary(out=file)


s.save(str(output_folder / f"{s.name}.snpy"))

In [ ]:
sn_file = output_folder / "SN2026atb_snoopy.txt"
#sn_file = output_folder / "SN2026atb.snpy"

s = get_sn(str(sn_file))
s.EBVgal = galactic_reddening[s.name]

s.choose_model("EBV_model2", stype="st")
s.fit()
s.summary()

s.plot(outfile=str(output_folder / f"{s.name}_fit.pdf"))

with (output_folder / f"{s.name}_summary.txt").open("w") as file:
    s.summary(out=file)


s.save(str(output_folder / f"{s.name}.snpy"))

In [ ]:
sn_file = output_folder / "SN2019tgm_snoopy.txt"
#sn_file = output_folder / "SN2026atb.snpy"

s = get_sn(str(sn_file))
s.EBVgal = galactic_reddening[s.name]

s.choose_model("EBV_model2", stype="st")
s.fit()
s.summary()

s.plot(outfile=str(output_folder / f"{s.name}_fit.pdf"))

with (output_folder / f"{s.name}_summary.txt").open("w") as file:
    s.summary(out=file)


s.save(str(output_folder / f"{s.name}.snpy"))

In [ ]:
sn_file = output_folder / "SN2020se_snoopy.txt"
#sn_file = output_folder / "SN2020se.snpy"

s = get_sn(str(sn_file))
s.EBVgal = galactic_reddening[s.name]

s.choose_model("EBV_model2", stype="st")
s.fit()
s.summary()

s.plot(outfile=str(output_folder / f"{s.name}_fit.pdf"))

with (output_folder / f"{s.name}_summary.txt").open("w") as file:
    s.summary(out=file)


s.save(str(output_folder / f"{s.name}.snpy"))